In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# 🧱 LangChat 心智模型 | Week11-Day1


## 📌 Capability Inventory：能力清查——代码里到底有几个"能力"？

---

### ━━━ 1. 今日核心问题 ━━━

**为什么 LangChat 代码里有三套"能力注册体系"，但 ADR 只定义了一个 Capability？**

前两周我们建立了一个清晰的概念：Capability 是平台原子能力，与行业正交（ADR-003）。SkillRelease 是唯一可部署制品（v2 Domain Model）。但在代码里打开一看——Capability Catalog、SkillRelease Registry、Capability Gateway 三套注册体系并行存在。Capability Catalog 只有 2 个条目，SkillRelease 有 10 个，Capability Gateway 又是另一套。

** Capability 这个词在代码里名不副实到了什么程度？**

---

### ━━━ 2. 人话解释 ━━━

Jason，你在 ERP 行业 26 年，一定见过这种情况：

公司规范说"供应商统一在 SRM 系统管理"，但实际运行中：
- 财务系统有一份供应商清单（因为要付款）
- 采购系统有一份供应商清单（因为要下订单）
- 仓库有一份供应商清单（因为要收货）
- SRM 系统也有一份（规范说的那个）

四份清单，四套 ID，四种状态。规范说"统一管理"，现实是"各管各的"。

LangChat 的 Capability 现状完全一样。v2 Domain Model 冻结了一个干净的 Capability 定义（Business Domain 层，受治理的可复用能力语义），但代码里：

| 注册体系 | 位置 | 条目数 | 实际角色 |
|---|---|---|---|
| **Capability Catalog** | `capability/catalog.py` | 2 | 元数据展示（纯信息性） |
| **SkillRelease Registry** | `skill_release/registry.py` | 10 | 实际执行入口 |
| **Capability Gateway** | `capability_gateway/` | 独立 W01 体系 | MCP 评估/评分 |

这不是 bug，这是**目标态和当前态的距离**。

---

### ━━━ 3. LangChat 架构位置 ━━━

在 LangChat 四层架构中，今天的发现涉及三层：


In [ ]:
Business Domain Layer
 └── Capability（目标态：受治理的可复用能力语义）
     ↑ 名不副实：代码里 Capability Catalog 只有 2 条，纯展示

Supply Chain Layer
 └── SkillRelease（目标态：唯一可部署制品）
     ↑ 实际承载：W01-W09 + workflow.execute 共 10 条

Runtime Layer
 └── Capability Gateway（运行时调用外部能力）
     ↑ 独立体系：W01 MCP Gateway 有自己的 contracts/evaluation

Operations Layer
 └── Catalog Projection（目标态：只读投影）
     ↑ 未实现：Capability Catalog 不是投影，是手写硬编码


**关键发现**：Capability Catalog 和 SkillRelease Registry 之间没有引用关系。Capability Catalog 不知道 SkillRelease 的存在，SkillRelease 也不查 Capability Catalog。两条线完全平行。

---

### ━━━ 4. ADR 依据 ━━━

**ADR-003 §2.2 硬约束**：
> Capability ID 中禁止出现行业词。Capability 清单 = `langchat/capability/catalog.py` 中的 `CapabilityRegistry` + `langchat/skill_release/registry.py`

ADR 明确把两个 Registry 并列列为 Capability 清单的来源。但实际代码：

- `catalog.py` 的 `CapabilityRegistry` 注册的是 `CapabilityDescriptor`（有 input_schema/output_schema/effects/approval_policy）
- `skill_release/registry.py` 的 `SkillReleaseRegistry` 注册的是 `SkillReleaseDescriptor`（有 effect_policy/human_review_gate/workflow_binding）

两个 Descriptor 结构不同，字段语义不同，但 ADR-003 把它们都算作"Capability 清单"。

**ADR-003 §2.2.3**：
> Application = Capability 子集 × Industry 标签

但代码里 W01-W09 都是"mall.*"前缀的行业 Skill——它们的 `skill_id` 是 `langchat.w01.ops.anomaly`、`langchat.w02.customer.escalation`，全都绑定 mall 场景。**这些不是 Capability（跨行业），是 Application 级别的打包**。

**ADR-001 平台定位**：
> LangChat 是企业 AI 应用平台，不是行业产品

但当前 SkillRelease Registry 里 9/10 个 Skill 是商业地产专属。平台级 Capability 只有 `langchat.workflow.execute` 一个。

**v2 Domain Model §6 Capability 定义**：
> Capability：受治理的可复用能力语义或 Provider 契约描述。不是执行入口，不是 SkillRelease。

代码事实：`catalog.py` 里的 Capability 确实不是执行入口（`runtime_binding = {}`），但它也不是"可复用能力语义"——它只是一个被掏空的元数据壳。

---

### ━━━ 5. 代码验证 ━━━

#### 5.1 Capability Catalog — 只有 2 个条目


In [ ]:
# catalog.py — _register_p0_capabilities()
_registry.register(CapabilityDescriptor(
    capability_id="langchat.knowledge.query",    # 知识查询
    capability_version="v1",
    runtime_binding={},                           # ← 空的！
    provider="langchat",
))

_registry.register(CapabilityDescriptor(
    capability_id="langchat.workflow.execute",    # 工作流执行
    capability_version="v1",
    runtime_binding={},                           # ← 也是空的！
    provider="langchat",
))


`runtime_binding` 曾经指向 in-process adapter 函数，E6 迁移后变成空对象。OpenSpec `capability-catalog` 明确说：**"its value SHALL be the empty object `{}`"**。

这意味着 Capability Catalog 的 2 个条目是**纯展示性元数据**——不能被执行，不指向任何实现。

#### 5.2 SkillRelease Registry — 10 个可执行 Skill


In [ ]:
langchat.w01.ops.anomaly          → mall-ops-daily-anomaly       (read_only)
langchat.w02.customer.escalation  → mall-customer-service-...    (read_only)
langchat.w03.brand.research       → mall-brand-research-profile  (read_only)
langchat.w04.leasing.funnel       → mall-leasing-funnel-weekly   (read_only)
langchat.w05.pilot.review         → mall-pilot-review-report     (read_only)
langchat.w06.ar.aging             → mall-ar-aging-alert          (conditional_write) ← 需要 FLAG
langchat.w07.ticket.sla           → mall-work-order-sla          (conditional_write) ← 需要 FLAG
langchat.w08.software.helpdesk    → mall-software-helpdesk       (read_only)
langchat.w09.internal.service     → mall-internal-service        (read_only)
langchat.workflow.execute         → bound_workflow               (read_only)


每个 SkillRelease 都有 `executor_fn`——它们是**真正的执行入口**。

#### 5.3 Capability Gateway — 第三套体系


In [ ]:
# capability_gateway/contracts.py
class CapabilityRequest(BaseModel):
    capability_id: str  # ← 必须在 W01_CAPABILITY_IDS 里


W01 MCP Gateway 有自己的 `capability_id` 校验、evaluation runner、holdout scorecard。它不查 Capability Catalog，也不查 SkillRelease Registry。

#### 5.4 Capability API 默认关闭


In [ ]:
# settings/__init__.py
CAPABILITY_API_ENABLED: bool = False  # ← 默认关闭


Capability API 的两个元数据端点（`/list_capabilities` 和 `/describe_capability`）默认不开放。需要显式设置 `CAPABILITY_API_ENABLED=true`。

---

### ━━━ 6. 商业地产映射（LangChat → MI CRE 场景） ━━━

把三套注册体系映射到 MI 商业地产场景：

| LangChat 注册体系 | MI CRE 对应 | 问题 |
|---|---|---|
| **Capability Catalog**（2 条，纯展示） | MI 集团能力目录（"我们有什么能力"的 PPT） | PPT 上写了 2 个，实际能干的远不止 |
| **SkillRelease Registry**（10 条，可执行） | 各商场的数字化员工花名册 | 9/10 是商业地产专属，不是平台通用 |
| **Capability Gateway** | MI 数据中台的 API 网关 | 独立体系，跟集团能力目录没有关系 |

**MI 场景下的"名不副实"**：

假设 MI 集团说"我们有一个'租户查询'能力"（Capability），但：
- Capability Catalog 里没有注册（只有 knowledge.query 和 workflow.execute）
- SkillRelease Registry 里有一个 `langchat.w03.brand.research`，但它做的是品牌调研不是租户查询
- Capability Gateway 里 W01 MCP 做的是 anomaly detection，也不是租户查询

**结论**：Capability 概念在目标态是清晰的，但当前代码里它没有承载实际业务能力发现和路由的职责。真正的能力发现在 SkillRelease Registry，而 SkillRelease 的命名已经泄漏了行业语义（w01-w09 全是 mall 场景）。

---

### ━━━ 7. 与传统方案比较 ━━━

| 维度 | 传统 ERP（如 SAP） | LangChat 当前 | LangChat 目标态 |
|---|---|---|---|
| 能力注册 | SAP 有统一的 Business Object Catalog | 三套体系并存 | 统一 Catalog Projection |
| 能力发现 | T-Code / Fiori App Catalog | SkillRelease `list_visible()` | Catalog Projection + Marketplace |
| 能力执行 | RFC / BAPI | SkillRelease invoke | SkillRelease invoke（不变） |
| 能力版本 | 难以多版本共存 | Capability 版本不可变 ✅ | 继续保持 |
| 能力废弃 | 无正式机制 | deprecate + successor ✅ | 继续保持 |
| 跨行业复用 | 一个 BAPI 写死行业逻辑 | 9/10 Skill 是 mall 专属 | Capability 与 Industry 正交 |

**LangChat 做对的两件事**：
1. **版本不可变**：Published Capability 的 7 个字段不可修改，新版本需要新注册。比 SAP 的 BAPI 版本管理强太多。
2. **废弃有 successor**：deprecate 时指定后继能力 + sunset_at，调用方收到 Deprecation header。

**LangChat 需要改进的三件事**：
1. **Catalog 要成为真正的投影**：不是手写 2 条，而是从 SkillRelease Registry 自动投影。
2. **W01-W09 要拆层**：把 mall 行业语义从 Skill ID 中剥离到 Industry 标签。
3. **Capability Gateway 要归入统一体系**：三套不能继续并行。

---

### ━━━ 8. 架构师思考题 ━━━

**问题：如果明天突然要接入第二个行业（比如医院），你第一步改什么？**

这不是假设。如果 LangChat 真的要做"企业 AI 应用平台"（ADR-001），第二个行业一定不是商业地产。

具体挑战：
1. `langchat.w01.ops.anomaly` 这个 Skill ID 里没有行业词（遵守了 ADR-003），但 `workflow_binding.workflow_id = "mall-ops-daily-anomaly"` 泄漏了行业。改 workflow_id 还是接受？
2. Capability Catalog 只有 2 条通用能力。新行业需要新的 knowledge.query 能力（比如 `langchat.medical.knowledge.query`？），但 ADR-003 禁止行业词进 Capability ID。怎么解？
3. SkillRelease 的 `visibility = "platform" | "tenant"`——新行业的 Skill 是 platform 可见还是 tenant 隔离？
4. 现有的 9 个 mall Skill 会不会被新行业误调？scope 校验够不够？

**提示**：答案不在代码里，在 ADR-003 的正交模型里。Capability 是平台层（跨行业），Application 是行业层。W01-W09 不是 Capability，它们是 Application 级别的 Skill。

---

### ━━━ 9. 我的理解变化 ━━━

**以前以为**：Capability Catalog 是 LangChat 的能力目录，列出了平台所有能力。SkillRelease 是能力的具体实现。

**现在知道**：
1. Capability Catalog 只有 2 条记录，`runtime_binding = {}`，是**被掏空的元数据壳**。它曾经有实际功能（指向 in-process adapter），E6 迁移后变成了纯展示。
2. SkillRelease Registry 才是**真正的执行入口**，10 个 Skill 各有 executor_fn，通过 `POST /v1/skill-releases/{skill_id}/invoke` 调用。
3. 三套注册体系不是设计缺陷，是**演进过程的中间态**。目标态 Domain Model 清楚地定义了 Catalog Projection（从 SkillRelease 自动投影），但当前代码还没走到那里。
4. W01-W09 的命名看似通用（`ops.anomaly`、`customer.escalation`），但 `workflow_binding` 和 `display_name` 全是 mall 场景——**Skill ID 遵守了正交约束的字母，但违反了精神**。
5. Capability API 默认关闭（`CAPABILITY_API_ENABLED = False`）——因为 Catalog 还没有准备好面对外部。

**一句话总结**：Capability 在目标态是"平台原子能力"，在代码态是"2 条空壳元数据 + 10 个行业 Skill + 1 套独立 Gateway"的过渡状态。Week 11 的任务就是量化这个 Gap。

---

### ━━━ 10. 明日连接 + Semantic Layer ━━━

**明日主题**：Gap Matrix — 目标态对象 vs 代码实现，逐个打分

今天清查了 Capability Inventory，发现了三套体系的距离。明天要把这个发现结构化：Week 11-Day2 将逐个对照 v2 Domain Model 的 25+ 个目标态对象，给每个对象打分（已实现 / 部分实现 / 未实现 / 方向偏离），最终输出一张 Gap Matrix。

**Semantic Layer 位置**：


In [ ]:
Ontology（本体论）
  └── Capability 概念定义 ← ADR-003 定义了正交模型
        ↓
Domain Model（域模型）
  └── Capability 对象规约 ← v2 Target Domain Model §6 BD-04
        ↓
Capability Catalog（能力目录）← 当前：2 条空壳
        ↓                        目标：Catalog Projection 自动投影
SkillRelease Registry（技能发布）← 当前：10 条可执行
        ↓
Execution（执行）← POST /v1/skill-releases/{skill_id}/invoke


**今天的核心认知**：Capability 概念在 Ontology 层面是清晰的（ADR-003），在 Domain Model 层面是冻结的（v2 Target Domain Model），但在代码层面还停留在 P0 最小实现——2 条元数据 + 10 个行业 Skill。从 P0 到目标态的路，就是 Week 11 要画出来的。
